# Part D (PM) — AI-Augmented: Algorithm Selection Decision Guide
**Day 33 | PM Session | Week 6**

Steps 9-12: Generate a decision guide for algorithm selection based on dataset characteristics, verify each recommendation, identify edge cases, and save as a personal reference.

---
## The Algorithm Selection Framework

*(Prompted, then verified and corrected against week-6 experience)*

```
START
 │
 ├─ Is the task CLASSIFICATION or REGRESSION?
 │
 ├─ CLASSIFICATION ──────────────────────────────────────────────────────┐
 │                                                                        │
 │   Q1: How many samples?                                               │
 │   ├─ < 100        → Naive Bayes / Logistic Regression (L1)           │
 │   ├─ 100 – 10K    → SVM (RBF or Linear), Random Forest, LR          │
 │   ├─ 10K – 100K   → Gradient Boosting / XGBoost, Random Forest      │
 │   └─ > 100K       → Logistic Regression, LinearSVC, Neural Networks  │
 │                                                                        │
 │   Q2: How many features?                                              │
 │   ├─ > samples    → Logistic Regression (L1), LinearSVC             │
 │   ├─ Moderate     → All algorithms viable                           │
 │   └─ Text / Sparse→ TF-IDF + LinearSVC or LogReg                   │
 │                                                                        │
 │   Q3: Is the boundary linear or non-linear?                          │
 │   ├─ Linear       → Logistic Regression, LinearSVC                  │
 │   └─ Non-linear   → SVM (RBF), Random Forest, GBM, KNN             │
 │                                                                        │
 │   Q4: Do you need probability output?                                │
 │   ├─ Yes          → Logistic Regression, GBM (calibrated), NB       │
 │   └─ No           → SVM, Random Forest, Decision Tree               │
 │                                                                        │
 │   Q5: Is interpretability required?                                  │
 │   ├─ High         → Logistic Regression, Decision Tree              │
 │   ├─ Medium       → Random Forest (feature importance)              │
 │   └─ Not needed   → SVM, GBM, XGBoost                              │
 │                                                                        │
 └────────────────────────────────────────────────────────────────────────┘
```

---
## Interactive Decision Helper

In [ ]:
def recommend_algorithm(
    n_samples: int,
    n_features: int,
    is_linear: bool,
    need_probability: bool,
    need_interpretability: bool,
    is_text: bool = False
) -> None:
    """
    Rule-based algorithm recommender for Week-6 algorithms.

    Parameters
    ----------
    n_samples             : Number of training samples
    n_features            : Number of features
    is_linear             : Is the expected decision boundary approximately linear?
    need_probability      : Do you need calibrated probability output?
    need_interpretability : Must the model be interpretable?
    is_text               : Is the data text (sparse)?
    """
    print('=' * 55)
    print('   ALGORITHM RECOMMENDATION')
    print('=' * 55)
    print(f'  Samples       : {n_samples}')
    print(f'  Features      : {n_features}')
    print(f'  Linear?       : {is_linear}')
    print(f'  Need prob?    : {need_probability}')
    print(f'  Interpret?    : {need_interpretability}')
    print(f'  Text/Sparse?  : {is_text}')
    print('-' * 55)

    recommendations = []
    avoid = []
    notes = []

    # --- Text shortcut ---
    if is_text:
        recommendations = ['LinearSVC (TF-IDF pipeline)', 'Logistic Regression (TF-IDF pipeline)']
        avoid = ['KNN (cosine in high-dim is unreliable)', 'Decision Tree (too many splits)']
        notes.append('For text, always use LinearSVC or LR with TF-IDF/CountVectorizer.')

    else:
        # --- Feature/sample ratio ---
        if n_features >= n_samples:
            recommendations += ['Logistic Regression (L1)', 'LinearSVC']
            avoid += ['Decision Tree', 'KNN', 'Gradient Boosting']
            notes.append('p >= n: use sparse linear models with regularisation.')

        # --- Sample size ---
        if n_samples < 200:
            recommendations += ['Naive Bayes', 'Logistic Regression (L2)']
            avoid += ['Gradient Boosting', 'XGBoost']
            notes.append('Very small n: prefer simple models; use LOOCV.')
        elif n_samples < 10_000:
            if is_linear:
                recommendations += ['Logistic Regression', 'LinearSVC']
            else:
                recommendations += ['SVM (RBF)', 'Random Forest']
        else:
            recommendations += ['Gradient Boosting', 'XGBoost', 'Random Forest']
            if is_linear:
                recommendations.append('Logistic Regression')

        # --- Probability ---
        if need_probability:
            if 'SVM (RBF)' in recommendations:
                recommendations.remove('SVM (RBF)')
                notes.append('SVM removed: set probability=True if SVM is needed (slower).')
            if 'Logistic Regression' not in recommendations:
                recommendations.append('Logistic Regression')

        # --- Interpretability ---
        if need_interpretability:
            recommendations = [r for r in recommendations
                               if r not in ['SVM (RBF)', 'Gradient Boosting', 'XGBoost']]
            recommendations = ['Logistic Regression', 'Decision Tree'] + recommendations
            notes.append('Interpretability needed: prefer LR (coefficients) or DT (tree plot).')

    # Deduplicate
    seen = set()
    recommendations = [r for r in recommendations if not (r in seen or seen.add(r))]

    print('  RECOMMENDED (in order):')
    for i, r in enumerate(recommendations[:4], 1):
        print(f'    {i}. {r}')
    if avoid:
        print('\n  AVOID:')
        for a in set(avoid):
            print(f'    ✗ {a}')
    if notes:
        print('\n  NOTES:')
        for n in notes:
            print(f'    → {n}')
    print('=' * 55)


# --- Test scenarios ---
print('Scenario 1: Medical diagnosis, 500 samples, 20 features, interpretability required')
recommend_algorithm(500, 20, is_linear=False, need_probability=True, need_interpretability=True)

print('\nScenario 2: Handwritten digit recognition, 1800 samples, 64 features')
recommend_algorithm(1800, 64, is_linear=False, need_probability=False, need_interpretability=False)

print('\nScenario 3: Text spam detection, 10K docs, ~50K TF-IDF features')
recommend_algorithm(10_000, 50_000, is_linear=True, need_probability=False, need_interpretability=False, is_text=True)

print('\nScenario 4: p >> n (gene expression): 80 samples, 500 features')
recommend_algorithm(80, 500, is_linear=True, need_probability=False, need_interpretability=False)

---
## 10-11 — Verification & Edge Cases the AI Missed

### Verified ✅
- Text → LinearSVC is correct (proven by academic literature and Part B results)
- p>>n → L1 regularisation is standard practice (LASSO, ElasticNet)
- Small n → prefer Naive Bayes / LR (validated in Q1 conceptual answer)
- Interpretability → LR / DT (medical/legal requirement is real)

### Edge Cases the AI Missed

| Edge Case | What to actually do |
|-----------|--------------------|
| **Imbalanced classes** | Use `class_weight='balanced'` in LR/SVM; SMOTE; adjust decision threshold; use F1/AUC not accuracy |
| **Mixed feature types** (numerical + categorical) | Random Forest or GBM handle natively; LR/SVM need encoding first |
| **Many irrelevant features** (feature noise) | Run SelectKBest or recursive feature elimination *before* model selection |
| **Ordinal targets** (e.g., 1-5 rating) | Ordinal regression or treat as multi-class carefully; standard classifiers assume equal class gap |
| **Online / streaming data** | SGDClassifier (online LR/SVM); standard batch models retrain from scratch |
| **Explainability for black-box** | SHAP values for RF/GBM — not just 'avoid' these models, but use SHAP to explain them |
| **Multi-label** (multiple correct labels) | Multi-label classifiers (`MultiLabelBinarizer` + `OneVsRestClassifier`); standard classifiers are multi-class not multi-label |

---
## 12 — Personal Algorithm Selection Reference Card

| Situation | First Choice | Second Choice | Avoid |
|-----------|-------------|---------------|-------|
| Quick baseline | Logistic Regression | Naive Bayes | — |
| Best accuracy (tabular) | XGBoost / GBM | Random Forest | Decision Tree (single) |
| Small dataset (<500) | LR (L2) or Naive Bayes | SVM (Linear) | GBM, XGBoost |
| High-dimensional (p>>n) | LR (L1) | LinearSVC | KNN, DT, GBM |
| Text classification | LinearSVC + TF-IDF | LogReg + TF-IDF | KNN |
| Need probability | LR / GBM | NB | SVM (use `probability=True` as fallback) |
| Need interpretability | Logistic Regression | Decision Tree | GBM, SVM |
| Non-linear boundary | SVM (RBF) | Random Forest | LR |
| Very large dataset (>100K) | LR / LinearSVC | LightGBM | SVM (RBF) |
| Similarity / recommendation | KNN + FAISS | — | GBM |
| Imbalanced classes | LR/SVM with class_weight | RF with class_weight | Accuracy metric |

> **Golden rule**: Always start with Logistic Regression as your baseline. It's fast, interpretable, and surprisingly competitive. Beat it before reaching for complexity.